In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["PATH"] = "/mnt/lustre-grete/usr/u12045/projects/LLAVA-Med/envs/lerobot/bin:" + os.environ.get("PATH", "")
os.environ["HF_HOME"] = "/mnt/lustre-grete/usr/u12045/vla/hf_cache"
os.environ["TMPDIR"] = "/mnt/lustre-grete/usr/u12045/vla/cache"
os.environ["PYTHONPATH"] = "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid:" + os.environ.get("PYTHONPATH", "")
import sys
sys.path.insert(0, "/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid")


import json
import pickle
from pathlib import Path

import torch

from lerobot.common.datasets.lerobot_dataset import LeRobotDatasetMetadata
from lerobot.common.policies.factory import make_policy
from lerobot.configs.policies import PreTrainedConfig

%load_ext autoreload
%autoreload 2

In [3]:
def display(tensor: torch.Tensor):
    if tensor.dtype == torch.bool:
        tensor = tensor.float()
    print(f"Shape: {tensor.shape}")
    print(f"Mean: {tensor.mean().item()}")
    print(f"Std: {tensor.std().item()}")
    print(f"Min: {tensor.min().item()}")
    print(f"Max: {tensor.max().item()}")

In [4]:
num_motors = 15
device = "cuda"
dataset_repo_id = "ducido/calvin_task_D_D_scale_50_lerobo_format"

ckpt_torch_dir = Path(f"/mnt/lustre-grete/usr/u12045/vla/duci/VLA-Humanoid/calvin_jaxcp_conversion_to_torch")
ckpt_jax_dir = Path(f"/mnt/lustre-grete/usr/u12045/vla/duci/openpi/checkpoints/pi0_calvin_50%_joint/pi0_calvin_50%_joint/30000")
save_dir = Path(f"/mnt/lustre-grete/usr/u12045/vla/duci/openpi/example_sample")

with open(save_dir / "example.pkl", "rb") as f:
    example = pickle.load(f)
with open(save_dir / "outputs.pkl", "rb") as f:
    outputs = pickle.load(f)
with open(save_dir / "noise.pkl", "rb") as f:
    jax_noise = pickle.load(f)
with open(save_dir / "jax_batch_norm.pkl", "rb") as f:
    jax_batch_norm = pickle.load(f)

with open(ckpt_jax_dir / f"assets/{dataset_repo_id}/norm_stats.json") as f:
    norm_stats = json.load(f)


In [5]:

# Create LeRobot batch from Jax
batch = {}
batch[f"image"] = torch.from_numpy(example['observation/image']).permute(2,0,1) / 255.0
batch[f"wrist_image"] = torch.from_numpy(example['observation/wrist_image']).permute(2,0,1) / 255.0
batch["state"] = torch.from_numpy(example["observation/state"])
batch["action"] = torch.from_numpy(outputs["actions"])
batch["task"] = example["prompt"]

# Batchify
for key in batch:
    if isinstance(batch[key], torch.Tensor):
        batch[key] = batch[key].unsqueeze(0)
    elif isinstance(batch[key], str):
        batch[key] = [batch[key]]
    else:
        raise ValueError(f"{key}, {batch[key]}")

# To device
for k in batch:
    if isinstance(batch[k], torch.Tensor):
        batch[k] = batch[k].to(device=device, dtype=torch.float32)


In [6]:

# Override stats
dataset_meta = LeRobotDatasetMetadata(dataset_repo_id)
dataset_meta.stats["state"]["mean"] = torch.tensor(
    norm_stats["norm_stats"]["state"]["mean"][:num_motors], dtype=torch.float32
)
dataset_meta.stats["state"]["std"] = torch.tensor(
    norm_stats["norm_stats"]["state"]["std"][:num_motors], dtype=torch.float32
)
dataset_meta.stats["joint_actions"]["mean"] = torch.tensor(
    norm_stats["norm_stats"]["actions"]["mean"][:8], dtype=torch.float32
)
dataset_meta.stats["joint_actions"]["std"] = torch.tensor(
    norm_stats["norm_stats"]["actions"]["std"][:8], dtype=torch.float32
)

The dataset you requested (ducido/calvin_task_D_D_scale_50_lerobo_format) is in 2.0 format.
While current version of LeRobot is backward-compatible with it, the version of your dataset still uses global
stats instead of per-episode stats. Update your dataset stats to the new format using this command:
```
python lerobot/common/datasets/v21/convert_dataset_v20_to_v21.py --repo-id=ducido/calvin_task_D_D_scale_50_lerobo_format
```

If you encounter a problem, contact LeRobot maintainers on [Discord](https://discord.com/invite/s3KuuzsPFb)
or open an [issue on GitHub](https://github.com/huggingface/lerobot/issues/new/choose).



In [7]:

cfg = PreTrainedConfig.from_pretrained(ckpt_torch_dir)
cfg.pretrained_path = ckpt_torch_dir
policy = make_policy(cfg, dataset_meta)

# loss_dict = policy.forward(batch, noise=noise, time=time_beta)
# loss_dict["loss"].backward()
# print("losses")
# display(loss_dict["losses_after_forward"])
# print("pi_losses")
# display(pi_losses)

# actions = []
# for _ in range(50):
#     action = policy.select_action(batch, noise=noise)
#     actions.append(action)
# actions = torch.stack(actions, dim=1)

2025-07-21 18:15:25.398484: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753114525.417245 1311675 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753114525.422659 1311675 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753114525.438367 1311675 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753114525.438389 1311675 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753114525.438392 1311675 computation_placer.cc:177] computation placer alr

load pretrained policy
Loading weights from local directory


In [8]:
policy.config.empty_cameras = 1

In [ ]:
import numpy as np
print(np.sum(jax_noise))
noise = torch.from_numpy(jax_noise).to(device=device, dtype=torch.float32)
actions, images, img_masks, state, lang_tokens, lang_masks, (prefix_embs, prefix_att_2d_masks, past_key_values) = policy.select_action_chunk_compare(batch, orinoise=noise)

print("actions")
display(actions)
print()
print("pi_actions")
pi_actions = batch["action"]
display(pi_actions)
print("atol=3e-2", torch.allclose(actions, pi_actions, atol=3e-2))
print("atol=2e-2", torch.allclose(actions, pi_actions, atol=2e-2))
print("atol=1e-2", torch.allclose(actions, pi_actions, atol=1e-2))

7.001541


actions
Shape: torch.Size([1, 50, 8])
Mean: 0.3534854054450989
Std: 1.0658634901046753
Min: -1.6075434684753418
Max: 1.7419759035110474

pi_actions
Shape: torch.Size([1, 50, 8])
Mean: 0.35365408658981323
Std: 1.0652574300765991
Min: -1.6050947904586792
Max: 1.7409871816635132
atol=3e-2 True
atol=2e-2 True
atol=1e-2 True


In [39]:
past_key_values[0]

{'key_states': tensor([[[[-0.7227,  1.8281,  1.4062,  ..., -3.2031, -0.4473, -0.3223]],
 
          [[ 0.3184,  2.1875,  3.5781,  ..., -3.1250, -1.0859, -0.5352]],
 
          [[ 1.8750,  0.5000,  0.3789,  ..., -2.3125, -0.6719,  1.0078]],
 
          ...,
 
          [[-0.3105,  0.1367,  0.2617,  ..., -1.8203, -0.0654,  5.6562]],
 
          [[-0.3105,  0.1367,  0.2617,  ..., -1.8203, -0.0654,  5.6562]],
 
          [[-0.3105,  0.1367,  0.2617,  ..., -1.8203, -0.0654,  5.6562]]]],
        device='cuda:0', dtype=torch.bfloat16),
 'value_states': tensor([[[[-7.6562e-01,  7.6562e-01, -2.9102e-01,  ..., -1.6016e-01,
            -6.6406e-01,  9.9609e-01]],
 
          [[-1.5391e+00,  9.0625e-01,  1.6211e-01,  ..., -6.5234e-01,
            -6.3965e-02,  5.8984e-01]],
 
          [[-2.0703e-01,  1.0107e-01, -2.9785e-02,  ...,  8.5449e-03,
             3.7994e-03,  9.6191e-02]],
 
          ...,
 
          [[-6.3438e+00, -1.5312e+00,  8.3203e-01,  ..., -2.5312e+00,
            -2.0938e+00,  

In [41]:
jax_batch_norm['kv_cache']

array([[[[[[-0.726562, 1.83594, 1.40625, ..., -3.20312, -0.449219,
            -0.3125]],

          [[0.318359, 2.1875, 3.57812, ..., -3.125, -1.09375,
            -0.53125]],

          [[1.875, 0.5, 0.375, ..., -2.3125, -0.675781, 1.00781]],

          ...,

          [[-0.310547, 0.134766, 0.263672, ..., -1.8125, -0.0673828,
            5.65625]],

          [[-0.310547, 0.134766, 0.263672, ..., -1.8125, -0.0673828,
            5.65625]],

          [[-0.310547, 0.134766, 0.263672, ..., -1.8125, -0.0673828,
            5.65625]]]],



        [[[[-0.820312, 0.154297, 0.574219, ..., 0.640625, 0.237305,
            1.1875]],

          [[-1.28906, 0.0361328, 0.714844, ..., 0.675781, -0.0461426,
            0.738281]],

          [[-0.0563965, -0.134766, 0.142578, ..., -0.46875, 0.355469,
            0.318359]],

          ...,

          [[-1.94531, -2.01562, -1.07812, ..., -0.408203, -0.119141,
            1.71094]],

          [[-1.94531, -2.01562, -1.07812, ..., -0.408203, -0.1191

In [137]:
import numpy as np
np.allclose(state.cpu().numpy(), jax_batch_norm['state'], atol=1e-3) # True 1e-4->False

True

In [148]:
# image
print(np.allclose(images[0].permute(0,2,3,1).cpu().numpy(), jax_batch_norm['images']["base_0_rgb"], atol=1e-7))
print(np.allclose(img_masks[0].cpu().numpy(), jax_batch_norm['image_masks']["base_0_rgb"], atol=1e-7))

True
True


In [149]:
# wrist image
print(np.allclose(images[1].permute(0,2,3,1).cpu().numpy(), jax_batch_norm['images']["left_wrist_0_rgb"], atol=1e-7))
print(np.allclose(img_masks[1].cpu().numpy(), jax_batch_norm['image_masks']["left_wrist_0_rgb"], atol=1e-7))

True
True


In [56]:
# blank image
print(np.allclose(images[2].permute(0,2,3,1).cpu().numpy(), jax_batch_norm['images']["right_wrist_0_rgb"], atol=1e-30))
print(np.allclose(img_masks[2].cpu().numpy(), jax_batch_norm['image_masks']["right_wrist_0_rgb"], atol=1e-7))

True
True


In [57]:
np.all(lang_tokens.cpu().numpy() == jax_batch_norm['tokenized_prompt']), np.all(lang_masks.cpu().numpy() == jax_batch_norm['tokenized_prompt_mask'])

(Array(True, dtype=bool), Array(True, dtype=bool))